In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_5_try_0.pickle

In [ ]:
%%PandasProfile
### cell 6 ###

def count_then_return_percent_18(df, col):
    # faster: use normalize in value_counts
    return df[col].value_counts(dropna=False, normalize=True).mul(100).round(1)


def combine_subset_of_data_from_multiple_years_18(question_of_interest, x_axis_title, include_2017=False):
    # map each year to its source DataFrame (no in-place modifications)
    df_map = {
        '2018': responses_df_2018_cell10,
        '2019': responses_df_2019_cell10,
        '2020': responses_df_2020,
        '2021': responses_df_2021,
        '2022': responses_df_2022_cell10,
    }
    if include_2017:
        # rename 2017 columns on-the-fly
        df_map['2017'] = responses_df_2017.rename(columns={
            'Country': question_of_interest,
            'CurrentJobTitleSelect': 'Select the title most similar to your current role (or most recent title if retired): - Selected Choice'
        })

    rows = []
    for year in sorted(df_map.keys()):
        df_y = df_map[year]
        # 2022: rename UK column and values
        if year == '2022':
            df_y = df_y.rename(columns={
                'United Kingdom (UK)': 'United Kingdom of Great Britain and Northern Ireland'
            })
            df_y = df_y.assign(**{
                question_of_interest: df_y[question_of_interest].replace({
                    'United Kingdom (UK)': 'United Kingdom of Great Britain and Northern Ireland'
                })
            })
        # 2017: standardize country values
        if year == '2017':
            df_y = df_y.assign(**{
                question_of_interest: df_y[question_of_interest].replace({
                    'United States': 'United States of America',
                    "People 's Republic of China": 'China',
                    'United Kingdom': 'United Kingdom of Great Britain and Northern Ireland'
                })
            })
        # collapse to subset_of_countries, label others 'Other'
        df_y = df_y.assign(**{
            question_of_interest: df_y[question_of_interest].where(
                df_y[question_of_interest].isin(subset_of_countries), 'Other'
            )
        })
        # compute percent and build year DataFrame
        pct = count_then_return_percent_18(df_y, question_of_interest).sort_index()
        df_pct = pct.reset_index().rename(columns={'index': question_of_interest, question_of_interest: 'percentage'})
        df_pct['year'] = year
        rows.append(df_pct)

    combined = pd.concat(rows, ignore_index=True)
    # rename the x-axis column if needed
    if x_axis_title:
        combined = combined.rename(columns={question_of_interest: x_axis_title})
    return combined

# prepare parameters
subset_of_countries = [
    'Other', 'India', 'United States of America', 'Brazil', 'Nigeria',
    'Pakistan', 'Japan', 'China', 'Egypt', 'Indonesia', 'Mexico',
    'Turkey', 'Russia'
]
question_of_interest_cell18 = 'In which country do you currently reside?'
# empty string for x-axis title as before
title_for_x_axis = ''

# get combined DataFrame
country_df_combined = combine_subset_of_data_from_multiple_years_18(
    question_of_interest_cell18,
    title_for_x_axis,
    include_2017=True
)
# sort and finalize naming
country_df_combined_cell18 = country_df_combined.sort_values(['year', 'percentage'], ascending=True)
# replace long UK name with shorter
country_df_combined_cell18 = country_df_combined_cell18.replace(
    'United Kingdom of Great Britain and Northern Ireland',
    'United Kingdom'
)
country_df_combined_cell18.info()